# 01 - Build CellPhoneDB inputs (clinical_outcome_v2)

**Gate 1.** Builds three input families under `<output_root>/inputs/`, verifies each one by reopening
it from disk, then stops for human review.

| Family | Location | Content |
|---|---|---|
| **A** all cells | `inputs/all/` | all 95,441 cells, all 19 `annot_6` types |
| **B** pooled balanced | `inputs/pooled_seed{0..4}/{good,poor}/` | Rule 2 then Rule 1; seed 0 is the main run, seeds 1-4 are stability replicates |
| **C** per patient | `inputs/per_patient/<SAMPLE>/` | one directory per patient, only types with >= 10 cells **in that patient** |

Rule order for family **B** (and only B):

1. **Rule 2 - cell-type retention.** Drop any cell type whose smaller outcome group falls below the
   pooled threshold. 15 of 19 types must survive; the 4 dropped must be exactly
   `Basal cells - Tumor`, `Fibroblast cells - CAF`, `Fibroblast cells - Inflammatory`, `Plasma cells`.
2. **Rule 1 - balancing.** Within each retained type, downsample each outcome group to
   `min(n_good, n_poor)` using `np.random.default_rng(seed)`.

Everything cheap and falsifiable (contract probe, Rule 2, per-patient retention) is asserted
**before** any gigabyte is written, so a contract breach fails in seconds rather than hours.

Patient lists are derived at runtime from `batch -> clinical_outcome`; no patient ID is hardcoded.
`adata.X` (dense z-scored float64, ~7.6 GiB) is never read - only `layers['counts']`.

In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
# ============================== CONFIG CELL (the only one) ==============================
import os, sys, json, shutil
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad
import scanpy as sc
import h5py

PROJECT_DIR = Path(os.environ.get(
    "CPDB_V2_DIR",
    f"{ACC_DATA_ROOT}/scripts/named_scripts/cellphoneDB_scripts/clinical_outcome_v2",
)).resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

CONFIG_PATH = PROJECT_DIR / "config.yaml"

from utils import (load_config, config_hash, sha256_file, write_manifest, log_jsonl,
                   derive_patient_outcomes, celltype_abundance_matrix, pooled_retained_celltypes,
                   balance_downsample, patient_retained_celltypes, core_celltypes,
                   validate_counts_matrix, write_cpdb_input)

CFG = load_config(CONFIG_PATH)
CFG_HASH = config_hash(CFG)

H5AD_PATH    = Path(CFG["input_h5ad"])
OUTPUT_ROOT  = Path(CFG["output_root"])
SAMPLE_COL   = CFG["sample_column"]
CELLTYPE_COL = CFG["cell_type_column"]
OUTCOME_COL  = CFG["outcome_column"]
COUNTS_LAYER = CFG["counts_layer"]
PATIENT_MIN_CELLS  = int(CFG["patient_celltype_min_cells"])
POOLED_MIN_PER_GRP = int(CFG["pooled_celltype_min_per_group"])
POOLED_SEEDS       = [int(s) for s in CFG["pooled_downsample_seeds"]]
PRIMARY_SEED       = POOLED_SEEDS[0]
CORE_MIN_PATIENTS  = int(CFG["secondary_core_min_patients"])
CPDB_SEED          = int(CFG.get("cpdb_debug_seed", 6))

# Expression scale CellPhoneDB v5 must receive. CPDB does not normalise internally and must not be
# given scaled/z-scored values; the legacy pipeline and the analysis plan both use log1p(CP10K).
COUNTS_CONTRACT = str(CFG.get("cpdb_counts_normalisation", "log1p_cp10k"))
TARGET_SUM      = float(CFG.get("normalize_target_sum", 1e4))
# Tolerance when re-deriving the per-cell library size from log1p values. The float32
# log1p/expm1 round trip on this matrix deviates by at most 0.007 counts in 10,000, so 1.0
# is three orders of magnitude of headroom and still catches a wrong or absent transform.
SCALE_ATOL      = 1.0

INPUTS_DIR    = OUTPUT_ROOT / "inputs"
AUDIT_DIR     = OUTPUT_ROOT / "audit"
LOG_DIR       = OUTPUT_ROOT / "logs"
GATE0_PATH    = AUDIT_DIR / "gate0_success.json"
GATE1_MD      = OUTPUT_ROOT / "gate1_summary.md"
MANIFEST_PATH = OUTPUT_ROOT / "MANIFEST.sha256"
EVENT_LOG     = LOG_DIR / "01_build_inputs_events.jsonl"

FORBIDDEN_SUBSTRINGS = ("cpdb_results_", "t1_cpdb_results", "t2_cpdb_results", "t3_cpdb_results",
                        "cellphoneDB_copy_backup")

# ---- Expected facts: ASSERTED, never imposed -----------------------------------------
EXPECTED = dict(
    n_cells=95441, n_celltypes=19, n_patients=24,
    n_good_patients=13, n_poor_patients=11,
    n_good_cells=46260, n_poor_cells=49181,
    pooled_dropped_celltypes=["Basal cells - Tumor", "Fibroblast cells - CAF",
                              "Fibroblast cells - Inflammatory", "Plasma cells"],
    pooled_n_retained=15,
    pooled_per_group_cells=42214,
    # Rule 1 balance target per retained type = min(n_good, n_poor). Verified against obs;
    # asserted element-by-element so a silent change in the annotation cannot pass.
    pooled_balance_target={
        "ADC - Tumor": 2249, "ADMEC - Tumor": 1633, "B cells": 213, "CD4+ T cells": 788,
        "CD8+ T cells": 413, "Endothelial cells": 938, "Epithelial cells": 2367,
        "Epithelial cells - Tumor": 28526, "Fibroblast cells": 2665,
        "Macrophages - AP/TAM": 276, "Macrophages - M2": 1045, "Mast cells": 43,
        "Memory B cells": 324, "Mural cells": 489, "Muscle cells": 245},
    # Retained under either threshold, but small enough that no claim should rest on it.
    pooled_smallest_retained="Mast cells",
    patient_retained_range=(5, 16),
    patient_retained_median=11,
    patient_weakest={"P25": 5, "P03": 6, "P02": 7, "P16": 7,
                     "P19": 8, "P18": 8},
    core_celltypes=["ADC - Tumor", "ADMEC - Tumor", "Endothelial cells",
                    "Epithelial cells - Tumor", "Fibroblast cells", "Macrophages - M2"],
)

RUN_STARTED = datetime.now(timezone.utc).isoformat()
CHUNK_ROWS = 10_000
META_CELLTYPE_COL = "cell_type"     # the column name CellPhoneDB requires in meta.tsv

In [ ]:
# ---- Recording helpers (utils.py owns all analysis logic) ----------------------------
CHECKS = []      # every assertion, blocking or not
BUILT = {}       # relative input dir -> verified stats
CHECKSUMS = {}   # relative path -> sha256


def log_event(record):
    log_jsonl(EVENT_LOG, {"ts": datetime.now(timezone.utc).isoformat(),
                          "notebook": "01_build_inputs", **record})


def check(cid, description, ok, detail=None, blocking=True):
    ok = bool(ok)
    entry = {"id": cid, "description": description, "passed": ok,
             "blocking": blocking, "detail": detail}
    CHECKS.append(entry)
    log_event({"event": "check", **entry})
    if not ok and blocking:
        raise AssertionError(f"[{cid}] {description} FAILED :: {detail}")
    return ok


def assert_safe_path(path):
    p = str(Path(path).resolve())
    bad = [s for s in FORBIDDEN_SUBSTRINGS if s in p]
    if bad:
        raise AssertionError(f"Refusing to touch protected location {p} (matched {bad})")
    if not p.startswith(str(OUTPUT_ROOT.resolve())):
        raise AssertionError(f"Refusing to write outside output_root: {p}")
    return Path(path)


def as_str_list(obj):
    if isinstance(obj, (pd.Series, pd.Index)):
        obj = obj.tolist()
    return sorted(str(x) for x in obj)


def register_checksum(path, digest=None):
    path = Path(path)
    rel = str(path.relative_to(OUTPUT_ROOT))
    CHECKSUMS[rel] = digest or sha256_file(path)
    return CHECKSUMS[rel]


LOG_DIR.mkdir(parents=True, exist_ok=True)
log_event({"event": "start", "config_hash": CFG_HASH, "started": RUN_STARTED,
           "downsample_seeds": POOLED_SEEDS, "primary_seed": PRIMARY_SEED,
           "cpdb_debug_seed": CPDB_SEED, "counts_contract": COUNTS_CONTRACT,
           "target_sum": TARGET_SUM})

## 0. Gate 0 must have passed

In [ ]:
check("A01.01", "gate0_success.json exists (notebook 00 completed)", GATE0_PATH.is_file(),
      detail=str(GATE0_PATH))
GATE0 = json.loads(GATE0_PATH.read_text())

check("A01.02", "Gate 0 passed", bool(GATE0.get("passed")), detail=GATE0.get("passed"))
check("A01.03", "Gate 0 used the same config hash", GATE0.get("config_hash") == CFG_HASH,
      detail={"gate0": GATE0.get("config_hash"), "now": CFG_HASH})

h5ad_sha = sha256_file(H5AD_PATH)
check("A01.04", "input H5AD checksum matches the one audited in Gate 0",
      GATE0.get("input_sha256", {}).get("input_h5ad") == h5ad_sha,
      detail={"gate0": GATE0.get("input_sha256", {}).get("input_h5ad"), "now": h5ad_sha})

CORE_CELLTYPES = as_str_list(GATE0.get("core_celltypes", []))
check("A01.05", "core cell-type set carried from Gate 0 is the expected 6",
      CORE_CELLTYPES == sorted(EXPECTED["core_celltypes"]), detail=CORE_CELLTYPES)

for d in (INPUTS_DIR, AUDIT_DIR, LOG_DIR):
    assert_safe_path(d).mkdir(parents=True, exist_ok=True)
preexisting_inputs = sorted(str(p.relative_to(OUTPUT_ROOT)) for p in INPUTS_DIR.rglob("*") if p.is_file())
check("A01.06", "inputs/ contains no pre-existing files", not preexisting_inputs,
      detail=preexisting_inputs[:10])

## 1. Load metadata, certify the raw counts, and build the CellPhoneDB payload

`layers['counts']` is read straight out of the HDF5 file as CSR float32 (93,307,416 non-zeros,
~750 MiB). The dense `X` group on disk is never opened.

**Expression scale.** `utils.write_cpdb_input` copies `layers[counts_layer]` into `X` *verbatim* -
it performs no normalisation. CellPhoneDB v5 does not normalise internally either, so the contract
has to be satisfied before the handoff. The raw counts are therefore certified as integral first
(`utils.validate_counts_matrix`, on the source matrix, while the values are still counts), and only
then transformed to `log1p(CP10K)` with `sc.pp.normalize_total(target_sum=1e4)` + `sc.pp.log1p()` -
the same transform the established pipeline (`sc_cpdb_all_sample.ipynb` /
`sc_cpdb_per_sample.ipynb`) fed to CellPhoneDB.

`normalize_total` is a per-cell rescale and `log1p` is elementwise, so normalising the full matrix
once is arithmetically identical to normalising each subset after splitting - and it guarantees that
a barcode carries identical values in every family it appears in.

`SOURCE` is the object handed to `utils.write_cpdb_input`; its `layers['counts']` is an alias of its
`X`, both holding the normalised payload. The z-scored `X` from disk is never read.


In [ ]:
try:
    from anndata.io import read_elem
except ImportError:                                    # anndata < 0.11 layout
    from anndata.experimental import read_elem

meta_view = ad.read_h5ad(H5AD_PATH, backed="r")
obs_full = meta_view.obs.copy()
var = meta_view.var.copy()
n_cells_total, n_genes = meta_view.shape
meta_view.file.close()

with h5py.File(H5AD_PATH, "r") as h5:
    raw_counts = read_elem(h5[f"layers/{COUNTS_LAYER}"])
raw_counts = raw_counts.tocsr() if sp.issparse(raw_counts) else sp.csr_matrix(raw_counts)

# Integrality is certified HERE, on the source matrix, while the values are still raw counts.
# After the normalisation below nothing in this notebook can make that claim again.
counts_report = validate_counts_matrix(raw_counts, name=COUNTS_LAYER)
check("A01.07", "source counts layer is sparse, finite, non-negative and integral (never densified)",
      counts_report["densified"] is False and counts_report["format"] == "csr", detail=counts_report)
check("A01.08", "counts shape matches the H5AD shape", raw_counts.shape == (n_cells_total, n_genes),
      detail=raw_counts.shape)
check("A01.09", "cell count is 95,441", n_cells_total == EXPECTED["n_cells"], detail=n_cells_total)

obs = pd.DataFrame({SAMPLE_COL: obs_full[SAMPLE_COL].astype(str),
                    CELLTYPE_COL: obs_full[CELLTYPE_COL].astype(str),
                    OUTCOME_COL: obs_full[OUTCOME_COL].astype(str)},
                   index=obs_full.index.astype(str))
check("A01.10", "barcodes are unique", obs.index.is_unique, detail=int(obs.index.nunique()))
check("A01.11", "19 cell types present", obs[CELLTYPE_COL].nunique() == EXPECTED["n_celltypes"],
      detail=as_str_list(obs[CELLTYPE_COL].unique()))
ALL_CELLTYPES = as_str_list(obs[CELLTYPE_COL].unique())

# ---- CellPhoneDB expression contract: raw counts -> CP10K -> log1p -------------------
SOURCE = ad.AnnData(X=raw_counts, obs=obs, var=pd.DataFrame(index=var.index.astype(str)))
sc.pp.normalize_total(SOURCE, target_sum=TARGET_SUM)
sc.pp.log1p(SOURCE)
SOURCE.layers[COUNTS_LAYER] = SOURCE.X    # alias, not a copy: this is the payload CPDB receives
del raw_counts                            # normalise_total worked in place; the name would mislead

check("A01.12", f"SOURCE carries the '{COUNTS_CONTRACT}' payload in layers['{COUNTS_LAYER}']",
      sp.issparse(SOURCE.X) and SOURCE.layers[COUNTS_LAYER] is SOURCE.X
      and SOURCE.shape == (n_cells_total, n_genes),
      detail={"sparse": bool(sp.issparse(SOURCE.X)), "shape": tuple(SOURCE.shape),
              "dtype": str(SOURCE.X.dtype), "nnz": int(SOURCE.X.nnz)})

_libsize = np.asarray(SOURCE.X.expm1().sum(axis=1)).ravel()
_dev = float(np.abs(_libsize - TARGET_SUM).max())
check("A01.62", f"every cell recovers a library size of {TARGET_SUM:g} under expm1 "
                f"(the log1p CP10K contract), max deviation <= {SCALE_ATOL}",
      _dev <= SCALE_ATOL, detail={"max_abs_deviation": _dev, "atol": SCALE_ATOL,
                                  "n_cells": int(_libsize.size)})
log_event({"event": "expression_contract_applied", "contract": COUNTS_CONTRACT,
           "target_sum": TARGET_SUM, "max_abs_deviation": _dev,
           "source_counts_validation": counts_report})
del _libsize


## 2. Derive the patient -> outcome mapping at runtime

In [ ]:
patient_outcomes = derive_patient_outcomes(obs, sample_col=SAMPLE_COL, outcome_col=OUTCOME_COL)

n_outcomes_per_patient = obs.groupby(SAMPLE_COL, observed=True)[OUTCOME_COL].nunique()
check("A01.13", "every patient maps to exactly one outcome",
      bool((n_outcomes_per_patient == 1).all()),
      detail=n_outcomes_per_patient[n_outcomes_per_patient != 1].to_dict())

PATIENTS = as_str_list(patient_outcomes.index)
GOOD_PATIENTS = as_str_list(patient_outcomes.index[patient_outcomes == "Good"])
POOR_PATIENTS = as_str_list(patient_outcomes.index[patient_outcomes == "Poor"])
n_good_cells = int((obs[OUTCOME_COL] == "Good").sum())
n_poor_cells = int((obs[OUTCOME_COL] == "Poor").sum())

check("A01.14", "24 patients derived", len(PATIENTS) == EXPECTED["n_patients"], detail=len(PATIENTS))
check("A01.15", "13 Good / 11 Poor patients",
      len(GOOD_PATIENTS) == EXPECTED["n_good_patients"]
      and len(POOR_PATIENTS) == EXPECTED["n_poor_patients"],
      detail={"good": len(GOOD_PATIENTS), "poor": len(POOR_PATIENTS)})
check("A01.16", "46,260 Good cells / 49,181 Poor cells",
      (n_good_cells, n_poor_cells) == (EXPECTED["n_good_cells"], EXPECTED["n_poor_cells"]),
      detail={"good": n_good_cells, "poor": n_poor_cells})
check("A01.17", "outcome levels are exactly {Good, Poor}",
      set(patient_outcomes.unique()) == {"Good", "Poor"}, detail=as_str_list(patient_outcomes.unique()))
check("A01.18", "patient lists agree with the Gate 0 record",
      GOOD_PATIENTS == as_str_list(GATE0["cohort"]["good_patients"])
      and POOR_PATIENTS == as_str_list(GATE0["cohort"]["poor_patients"]),
      detail={"good": GOOD_PATIENTS, "poor": POOR_PATIENTS})

log_event({"event": "patients_derived", "good": GOOD_PATIENTS, "poor": POOR_PATIENTS,
           "n_good_cells": n_good_cells, "n_poor_cells": n_poor_cells})
patient_outcomes.to_frame()

## 3. Contract probe: what actually lands on disk?

`utils.write_cpdb_input` writes whatever `layers[counts_layer]` holds, so the probe re-reads a
200-cell throwaway input from disk and classifies its expression scale independently of the
in-memory object. It asserts the `log1p_cp10k` contract, the `cell_type` column name CellPhoneDB
requires, and barcode order - **before** any real input is built, so a contract breach fails in
seconds rather than after hours of writing. The probe directory is deleted immediately afterwards.

`validate=False` is passed to `write_cpdb_input` throughout this notebook: its internal validator
asserts *integral raw counts*, which the normalised payload is deliberately no longer. Integrality
was certified on the source matrix in section 1; the written files are certified against the
normalisation contract instead, chunk by chunk, in section 7.


In [ ]:
def classify_expression_scale(counts_h5ad, target_sum=TARGET_SUM, max_rows=CHUNK_ROWS):
    """Return 'raw_counts' | 'log1p_cp10k' | 'cp10k' | 'unknown' for a written counts.h5ad."""
    a = ad.read_h5ad(counts_h5ad, backed="r")
    try:
        ch = a.X[0:min(a.n_obs, max_rows)]
        ch = ch.to_memory() if hasattr(ch, "to_memory") else ch
        ch = ch if sp.issparse(ch) else sp.csr_matrix(ch)
        data = ch.data
        if data.size == 0:
            return "unknown", {}
        integral = bool(np.array_equal(data, np.floor(data)))
        rs_raw = np.asarray(ch.sum(axis=1)).ravel()
        rs_exp = np.asarray(ch.expm1().sum(axis=1)).ravel()
        stats = {"min": float(data.min()), "max": float(data.max()), "integral": integral,
                 "rowsum_median": float(np.median(rs_raw)),
                 "expm1_rowsum_median": float(np.median(rs_exp))}
        if integral and float(data.min()) >= 1.0:
            return "raw_counts", stats
        if np.allclose(rs_exp, target_sum, atol=1.0):
            return "log1p_cp10k", stats
        if np.allclose(rs_raw, target_sum, atol=1.0):
            return "cp10k", stats
        return "unknown", stats
    finally:
        if getattr(a, "file", None) is not None:
            a.file.close()


probe_dir = assert_safe_path(AUDIT_DIR / "_utils_contract_probe")
if probe_dir.exists():
    shutil.rmtree(probe_dir)
probe_cells = obs.index[:200]
probe_res = write_cpdb_input(SOURCE, probe_cells, probe_dir, CELLTYPE_COL,
                             counts_layer=COUNTS_LAYER, validate=False)
probe_meta = pd.read_csv(probe_dir / "meta.tsv", sep="\t", index_col=0)
observed_scale, scale_stats = classify_expression_scale(probe_dir / "counts.h5ad")

check("A01.19", "write_cpdb_input produces counts.h5ad and meta.tsv",
      Path(probe_res["counts_h5ad"]).is_file() and Path(probe_res["meta_tsv"]).is_file(),
      detail={k: probe_res[k] for k in ("counts_h5ad", "meta_tsv", "n_cells", "n_genes")})
check("A01.20", f"meta.tsv carries exactly one metadata column named '{META_CELLTYPE_COL}'",
      list(probe_meta.columns) == [META_CELLTYPE_COL], detail=list(probe_meta.columns))
check("A01.21", "meta.tsv row order matches the requested barcode order",
      pd.Index(probe_meta.index.astype(str)).equals(pd.Index(probe_cells.astype(str))),
      detail={"n_meta": len(probe_meta), "n_requested": len(probe_cells)})
check("A01.22", f"written expression scale satisfies the '{COUNTS_CONTRACT}' contract",
      observed_scale == COUNTS_CONTRACT,
      detail={"config_cpdb_counts_normalisation": COUNTS_CONTRACT,
              "observed_scale": observed_scale, "stats": scale_stats,
              "applied_by": ("this notebook, section 1 (sc.pp.normalize_total(target_sum="
                             f"{TARGET_SUM:g}) then sc.pp.log1p) -- utils.write_cpdb_input "
                             "normalises nothing, it copies layers[counts_layer] verbatim"),
              "on_failure": (
                  "Do NOT relax this check. Either the normalisation in section 1 did not run, or "
                  "utils.write_cpdb_input no longer writes layers[counts_layer] verbatim. "
                  "CellPhoneDB v5 does not normalise internally, so shipping raw counts here would "
                  "silently change every mean, p-value and interaction score downstream.")})

shutil.rmtree(probe_dir)
{"observed_scale": observed_scale, **scale_stats}

## 4. Family B, Rule 2 - cell-type retention (derived and asserted before anything is written)

A cell type enters the pooled comparison only if **both** outcome groups exceed
`pooled_celltype_min_per_group` (= 10) cells. The realised retained/dropped split is asserted **by
name**; the threshold is never tuned to reach the expected answer.

**The threshold is exclusive, and that matters for exactly one cell type.** `Plasma cells` sits on
the boundary at 10 Good / 157 Poor. `utils.pooled_retained_celltypes` implements the *inclusive*
predicate `min(n_good, n_poor) >= min_per_group`, so it is called here with `min_per_group + 1` to
obtain the exclusive form. Both readings are computed and recorded:

| reading | predicate | retained | cells per group |
|---|---|---|---|
| inclusive | `min >= 10` | 16 | 42,224 |
| **exclusive (used)** | `min > 10` | **15** | **42,214** |

> **This is a deliberate resolution of a boundary case, not a transcription of the analysis notes. Do not
> "correct" it back to `>=`.** The earlier analysis notes contradict themselves: they state the rule as
> `min < 10 -> drop` (which would *retain* `Plasma cells`) while simultaneously listing `Plasma
> cells` among the drops and quoting 15 types / 42,214 cells per group (which requires `> 10`). The
> difference is exactly 10 cells - `Plasma cells`' own minimum. The decision taken here is to **drop
> Plasma cells**, and the exclusive threshold is the reading that makes all three quoted facts true
> at once.

The rationale on its own terms: balancing `Plasma cells` would downsample Poor from 157 to 10, so the
entire comparison for that type would rest on 10 cells per group - below what either plan considers
stable for CellPhoneDB - and Rule 2 exists precisely to remove presence/absence artefacts.

`Mast cells` (45 Good / 43 Poor) is retained under either threshold, but it is the smallest retained
type by an order of magnitude; `gate1_summary.md` flags that no claim should rest on it.

The decision is recorded in `RULE2` and reproduced in `gate1_summary.md` rather than left implicit.


In [ ]:
ct_by_outcome = (pd.crosstab(obs[CELLTYPE_COL], obs[OUTCOME_COL])
                 .reindex(columns=["Good", "Poor"], fill_value=0).astype(int))
ct_by_outcome["min_group"] = ct_by_outcome.min(axis=1)

# utils.pooled_retained_celltypes keeps a type when min(n_good, n_poor) >= min_per_group.
# The specification's threshold is EXCLUSIVE, so the inclusive utility is called with +1.
retained_inclusive = as_str_list(pooled_retained_celltypes(
    obs, celltype_col=CELLTYPE_COL, outcome_col=OUTCOME_COL, min_per_group=POOLED_MIN_PER_GRP))
retained_types = as_str_list(pooled_retained_celltypes(
    obs, celltype_col=CELLTYPE_COL, outcome_col=OUTCOME_COL, min_per_group=POOLED_MIN_PER_GRP + 1))
dropped_types = as_str_list(set(ALL_CELLTYPES) - set(retained_types))
boundary_types = as_str_list(ct_by_outcome.index[ct_by_outcome["min_group"] == POOLED_MIN_PER_GRP])

RULE2 = {
    "threshold": POOLED_MIN_PER_GRP,
    "predicate_used": f"min(n_good, n_poor) > {POOLED_MIN_PER_GRP}",
    "utils_call": f"pooled_retained_celltypes(..., min_per_group={POOLED_MIN_PER_GRP + 1})",
    "boundary_celltypes": boundary_types,
    "inclusive": {"predicate": f"min >= {POOLED_MIN_PER_GRP}",
                  "n_retained": len(retained_inclusive),
                  "cells_per_group": int(ct_by_outcome.loc[retained_inclusive, "min_group"].sum())},
    "exclusive": {"predicate": f"min > {POOLED_MIN_PER_GRP}",
                  "n_retained": len(retained_types),
                  "cells_per_group": int(ct_by_outcome.loc[retained_types, "min_group"].sum())},
    "retained": retained_types,
    "dropped": dropped_types,
}

check("A01.23", "Rule 2 drops exactly the 4 specified cell types (by name)",
      dropped_types == sorted(EXPECTED["pooled_dropped_celltypes"]),
      detail={"dropped": dropped_types, "expected": sorted(EXPECTED["pooled_dropped_celltypes"]),
              "rule2": RULE2})
check("A01.24", "Rule 2 retains 15 of 19 cell types",
      len(retained_types) == EXPECTED["pooled_n_retained"],
      detail={"n_retained": len(retained_types), "retained": retained_types})
check("A01.25", f"every retained type EXCEEDS {POOLED_MIN_PER_GRP} cells in BOTH outcome groups",
      bool((ct_by_outcome.loc[retained_types, ["Good", "Poor"]].min(axis=1)
            > POOLED_MIN_PER_GRP).all()),
      detail=ct_by_outcome.loc[retained_types, "min_group"].to_dict())
check("A01.63", "the inclusive and exclusive readings differ only by the boundary cell types",
      set(retained_inclusive) - set(retained_types) == set(boundary_types)
      and set(retained_types) <= set(retained_inclusive),
      detail={"boundary": boundary_types,
              "inclusive_only": as_str_list(set(retained_inclusive) - set(retained_types)),
              "inclusive_cells_per_group": RULE2["inclusive"]["cells_per_group"],
              "exclusive_cells_per_group": RULE2["exclusive"]["cells_per_group"]})
check("A01.26", "every core cell type survives Rule 2", set(CORE_CELLTYPES) <= set(retained_types),
      detail=as_str_list(set(CORE_CELLTYPES) - set(retained_types)))

BALANCE_TARGET = ct_by_outcome.loc[retained_types, "min_group"].astype(int)
POOLED_PER_GROUP = int(BALANCE_TARGET.sum())
check("A01.27", "the balanced design yields 42,214 cells per outcome group",
      POOLED_PER_GROUP == EXPECTED["pooled_per_group_cells"],
      detail={"per_group": POOLED_PER_GROUP, "expected": EXPECTED["pooled_per_group_cells"],
              "total": 2 * POOLED_PER_GROUP,
              "fraction_of_all_cells": round(2 * POOLED_PER_GROUP / EXPECTED["n_cells"], 4)})

check("A01.68", "the Rule 1 balance target matches the verified per-type minima exactly",
      {k: int(v) for k, v in BALANCE_TARGET.items()} == EXPECTED["pooled_balance_target"],
      detail={"realised": {k: int(v) for k, v in BALANCE_TARGET.items()},
              "expected": EXPECTED["pooled_balance_target"],
              "differing": {k: {"realised": int(BALANCE_TARGET.get(k, -1)),
                                "expected": v}
                            for k, v in EXPECTED["pooled_balance_target"].items()
                            if int(BALANCE_TARGET.get(k, -1)) != v}})
check("A01.69", f"{EXPECTED['pooled_smallest_retained']} is the smallest retained type "
                "(retained under either threshold; flagged as fragile, not dropped)",
      BALANCE_TARGET.idxmin() == EXPECTED["pooled_smallest_retained"],
      detail={"smallest": str(BALANCE_TARGET.idxmin()), "n": int(BALANCE_TARGET.min())})

log_event({"event": "rule2_resolved", **RULE2})
ct_by_outcome.assign(retained=lambda d: d.index.isin(retained_types),
                     balance_target=lambda d: d.index.map(BALANCE_TARGET)).sort_values("min_group")


## 5. Family B, Rule 1 - balanced downsampling for seeds 0-4

Seed 0 is the primary pooled run; seeds 1-4 are stability replicates. Every seed must reproduce the
same retained cell types and the same 42,214-per-group size while selecting different cells.

In [ ]:
seed_selections = {}
pooled_tables = {}

for seed in POOLED_SEEDS:
    sel = pd.Index(balance_downsample(obs, celltype_col=CELLTYPE_COL, outcome_col=OUTCOME_COL,
                                      keep_types=retained_types, seed=seed).astype(str))
    check(f"A01.28.s{seed}", f"seed {seed}: selection is a subset of the source barcodes with no duplicates",
          sel.is_unique and set(sel) <= set(obs.index), detail={"n_selected": len(sel)})

    sub = obs.loc[sel]
    tab = (pd.crosstab(sub[CELLTYPE_COL], sub[OUTCOME_COL])
           .reindex(index=retained_types, columns=["Good", "Poor"], fill_value=0).astype(int))
    check(f"A01.29.s{seed}", f"seed {seed}: every retained type is balanced to min(n_good, n_poor)",
          bool((tab["Good"] == BALANCE_TARGET).all() and (tab["Poor"] == BALANCE_TARGET).all()),
          detail=tab.assign(target=BALANCE_TARGET).to_dict("index"))
    check(f"A01.30.s{seed}", f"seed {seed}: 42,214 Good and 42,214 Poor cells",
          int((sub[OUTCOME_COL] == "Good").sum()) == EXPECTED["pooled_per_group_cells"]
          and int((sub[OUTCOME_COL] == "Poor").sum()) == EXPECTED["pooled_per_group_cells"],
          detail={"good": int((sub[OUTCOME_COL] == "Good").sum()),
                  "poor": int((sub[OUTCOME_COL] == "Poor").sum())})
    check(f"A01.31.s{seed}", f"seed {seed}: no dropped cell type survives the downsample",
          as_str_list(sub[CELLTYPE_COL].unique()) == retained_types,
          detail=as_str_list(set(sub[CELLTYPE_COL].unique()) & set(dropped_types)))
    seed_selections[seed] = sel
    pooled_tables[seed] = tab

check("A01.32", "stability seeds select different cells from the primary seed",
      all(set(seed_selections[s]) != set(seed_selections[PRIMARY_SEED])
          for s in POOLED_SEEDS if s != PRIMARY_SEED),
      detail={s: len(set(seed_selections[s]) ^ set(seed_selections[PRIMARY_SEED]))
              for s in POOLED_SEEDS if s != PRIMARY_SEED})
check("A01.33", "all seeds produce the same per-group size",
      len({int((obs.loc[v, OUTCOME_COL] == "Good").sum()) for v in seed_selections.values()}) == 1,
      detail={s: int((obs.loc[v, OUTCOME_COL] == "Good").sum()) for s, v in seed_selections.items()})

pooled_tables[PRIMARY_SEED].assign(target=BALANCE_TARGET)

## 6. Family C - per-patient retention (derived and asserted before anything is written)

Within a patient, a cell type is retained when it has at least `patient_celltype_min_cells` cells in
**that** patient. The previously observed 5-16 range and median of 11 are compared, not enforced: a
deviation is flagged for investigation rather than tuned away.

In [ ]:
abundance_ct = celltype_abundance_matrix(obs, sample_col=SAMPLE_COL, celltype_col=CELLTYPE_COL)
patient_retained = {p: as_str_list(patient_retained_celltypes(
                        obs, p, PATIENT_MIN_CELLS,
                        sample_col=SAMPLE_COL, celltype_col=CELLTYPE_COL))
                    for p in PATIENTS}
direct_retained = {p: as_str_list(abundance_ct.index[abundance_ct[p] >= PATIENT_MIN_CELLS])
                   for p in PATIENTS}

check("A01.34", "utils.patient_retained_celltypes matches the abundance-matrix threshold",
      patient_retained == direct_retained,
      detail={p: {"utils": patient_retained[p], "abundance": direct_retained[p]}
              for p in PATIENTS if patient_retained[p] != direct_retained[p]})
check("A01.35", "all 24 patients have at least one retained cell type",
      len(patient_retained) == EXPECTED["n_patients"]
      and all(len(v) >= 1 for v in patient_retained.values()),
      detail={p: len(v) for p, v in patient_retained.items() if len(v) < 1})

n_ret = pd.Series({p: len(v) for p, v in patient_retained.items()}).sort_values()
check("A01.36", "per-patient retained-type counts fall in the previously observed 5-16 range",
      int(n_ret.min()) >= EXPECTED["patient_retained_range"][0]
      and int(n_ret.max()) <= EXPECTED["patient_retained_range"][1],
      detail={"min": int(n_ret.min()), "max": int(n_ret.max())}, blocking=False)
check("A01.37", "per-patient median retained-type count is 11 (investigate if not)",
      float(n_ret.median()) == float(EXPECTED["patient_retained_median"]),
      detail={"median": float(n_ret.median())}, blocking=False)
check("A01.38", "the six weakest patients match the previously observed counts",
      all(int(n_ret.get(p, -1)) == v for p, v in EXPECTED["patient_weakest"].items()),
      detail={p: int(n_ret.get(p, -1)) for p in EXPECTED["patient_weakest"]}, blocking=False)
check("A01.39", "every patient retains >= 2 cell types (fewer yields no interaction pair)",
      bool((n_ret >= 2).all()), detail={p: int(v) for p, v in n_ret.items() if v < 2}, blocking=False)

n_ret.rename("n_retained_celltypes").to_frame()

## 7. Builder and verifier

`build_input` hands the barcode list to `utils.write_cpdb_input` (which subsets, writes
`counts.h5ad` + `meta.tsv` and returns checksums), writes `retained_celltypes.json`, then reopens the
directory from disk and certifies it. The certified artefact is the file on disk, not the in-memory
object.

In [ ]:
def verify_input(out_dir, label, expected_barcodes, expected_celltypes,
                 expected_samples=None, expected_outcomes=None, min_cells_per_type=None):
    """Reopen a written input directory and certify it. Raises on any discrepancy."""
    out_dir = Path(out_dir)
    tag = f"V[{label}]"
    counts_path, meta_path = out_dir / "counts.h5ad", out_dir / "meta.tsv"

    a = ad.read_h5ad(counts_path, backed="r")
    try:
        meta = pd.read_csv(meta_path, sep="\t", index_col=0)
        meta.index = meta.index.astype(str)
        obs_names = pd.Index(a.obs_names.astype(str))
        expected_barcodes = pd.Index(pd.Index(expected_barcodes).astype(str))

        if META_CELLTYPE_COL not in meta.columns:
            raise AssertionError(f"{tag}: meta.tsv lacks a '{META_CELLTYPE_COL}' column "
                                 f"(has {list(meta.columns)})")
        if not obs_names.equals(meta.index):
            first = next((i for i, (x, y) in enumerate(zip(obs_names, meta.index)) if x != y), None)
            raise AssertionError(f"{tag}: counts.obs_names != meta.index (order-sensitive); "
                                 f"n_counts={len(obs_names)} n_meta={len(meta.index)} "
                                 f"first_difference_at={first}")
        if not obs_names.equals(expected_barcodes):
            raise AssertionError(f"{tag}: written barcodes differ from the requested selection "
                                 f"(written={len(obs_names)}, expected={len(expected_barcodes)}, "
                                 f"unexpected={len(obs_names.difference(expected_barcodes))})")
        if not obs_names.is_unique:
            raise AssertionError(f"{tag}: duplicate barcodes in the written input")

        written_types = as_str_list(meta[META_CELLTYPE_COL].unique())
        if written_types != as_str_list(expected_celltypes):
            raise AssertionError(f"{tag}: cell types written {written_types} != expected "
                                 f"{as_str_list(expected_celltypes)}")
        if not np.array_equal(meta[META_CELLTYPE_COL].astype(str).to_numpy(),
                              obs.loc[obs_names, CELLTYPE_COL].to_numpy()):
            raise AssertionError(f"{tag}: meta cell_type labels do not match the source annotation")

        src_samples = as_str_list(obs.loc[obs_names, SAMPLE_COL].unique())
        src_outcomes = as_str_list(obs.loc[obs_names, OUTCOME_COL].unique())
        if expected_samples is not None and src_samples != as_str_list(expected_samples):
            raise AssertionError(f"{tag}: samples {src_samples} != expected {as_str_list(expected_samples)}")
        if expected_outcomes is not None and src_outcomes != as_str_list(expected_outcomes):
            raise AssertionError(f"{tag}: outcomes {src_outcomes} != expected {as_str_list(expected_outcomes)}")

        per_type = meta[META_CELLTYPE_COL].value_counts().astype(int)
        if min_cells_per_type is not None and int(per_type.min()) < int(min_cells_per_type):
            raise AssertionError(f"{tag}: a retained cell type is below the {min_cells_per_type}-cell "
                                 f"threshold: {per_type[per_type < min_cells_per_type].to_dict()}")

        # ---- chunk-wise matrix checks; never densified in full ----
        x_min, x_max, off_scale, is_sparse = np.inf, -np.inf, 0, True
        for start in range(0, a.n_obs, CHUNK_ROWS):
            stop = min(start + CHUNK_ROWS, a.n_obs)
            ch = a.X[start:stop]
            ch = ch.to_memory() if hasattr(ch, "to_memory") else ch
            is_sparse &= bool(sp.issparse(ch))
            ch = ch if sp.issparse(ch) else sp.csr_matrix(ch)
            d = ch.data
            if d.size:
                if not np.isfinite(d).all():
                    raise AssertionError(f"{tag}: non-finite values in X at rows {start}:{stop}")
                if (d < 0).any():
                    raise AssertionError(f"{tag}: negative values in X at rows {start}:{stop}")
                x_min, x_max = min(x_min, float(d.min())), max(x_max, float(d.max()))
            if COUNTS_CONTRACT == "log1p_cp10k":
                rs = np.asarray(ch.expm1().sum(axis=1)).ravel()
                off_scale += int((np.abs(rs - TARGET_SUM) > SCALE_ATOL).sum())
            elif COUNTS_CONTRACT == "raw_counts":
                validate_counts_matrix(ch, name=f"{label}[{start}:{stop}]")
                off_scale += int((np.asarray(ch.sum(axis=1)).ravel() == 0).sum())
            del ch
        if off_scale:
            raise AssertionError(f"{tag}: {off_scale} cells violate the '{COUNTS_CONTRACT}' "
                                 "expression contract in the written counts.h5ad")
        if not is_sparse:
            raise AssertionError(f"{tag}: written matrix is dense; sparsity must be preserved")
        n_written, n_vars = int(a.n_obs), int(a.n_vars)
    finally:
        if getattr(a, "file", None) is not None:
            a.file.close()

    return {"label": label, "dir": str(out_dir.relative_to(OUTPUT_ROOT)),
            "n_cells": n_written, "n_genes": n_vars, "n_celltypes": len(written_types),
            "celltypes": written_types, "samples": src_samples, "outcomes": src_outcomes,
            "cells_per_celltype": per_type.sort_index().to_dict(),
            "x_min": None if x_min is np.inf else float(x_min),
            "x_max": None if x_max is -np.inf else float(x_max),
            "expression_contract": COUNTS_CONTRACT}


def build_input(barcodes, out_dir, label, expected_celltypes=None, expected_samples=None,
                expected_outcomes=None, min_cells_per_type=None, provenance=None):
    """Write one CellPhoneDB input directory via utils.write_cpdb_input, then certify it."""
    out_dir = assert_safe_path(out_dir)
    barcodes = pd.Index(pd.Index(barcodes).astype(str))

    # validate=False: the internal validator asserts integral RAW counts, which the normalised
    # CPDB payload deliberately is not. Integrality was certified on the source matrix in
    # section 1; verify_input below certifies the written file against the log1p CP10K contract.
    res = write_cpdb_input(SOURCE, barcodes, out_dir, CELLTYPE_COL,
                           counts_layer=COUNTS_LAYER, validate=False)
    retained = as_str_list(obs.loc[barcodes, CELLTYPE_COL].unique())
    (out_dir / "retained_celltypes.json").write_text(json.dumps({
        "label": label,
        "retained_celltypes": retained,
        "n_retained_celltypes": len(retained),
        "n_cells": int(len(barcodes)),
        "cells_per_celltype": {k: int(v) for k, v in res["cell_type_counts"].items()},
        "samples": as_str_list(obs.loc[barcodes, SAMPLE_COL].unique()),
        "outcomes": as_str_list(obs.loc[barcodes, OUTCOME_COL].unique()),
        "expression_contract": COUNTS_CONTRACT,
        "config_hash": CFG_HASH,
        "provenance": provenance or {},
    }, indent=2))

    stats = verify_input(out_dir, label, expected_barcodes=barcodes,
                         expected_celltypes=expected_celltypes or retained,
                         expected_samples=expected_samples, expected_outcomes=expected_outcomes,
                         min_cells_per_type=min_cells_per_type)
    register_checksum(out_dir / "counts.h5ad", res["counts_h5ad_sha256"])
    register_checksum(out_dir / "meta.tsv", res["meta_tsv_sha256"])
    register_checksum(out_dir / "retained_celltypes.json")
    stats["counts_h5ad_sha256"] = res["counts_h5ad_sha256"]
    stats["meta_tsv_sha256"] = res["meta_tsv_sha256"]
    BUILT[str(out_dir.relative_to(OUTPUT_ROOT))] = stats
    log_event({"event": "built", "label": label, "dir": stats["dir"],
               **{k: v for k, v in stats.items() if k not in ("cells_per_celltype", "dir")}})
    return stats

## 8. Family A - all cells (`inputs/all/`)

In [ ]:
stats_all = build_input(
    barcodes=obs.index, out_dir=INPUTS_DIR / "all", label="all",
    expected_celltypes=ALL_CELLTYPES, expected_samples=PATIENTS, expected_outcomes=["Good", "Poor"],
    provenance={"family": "A", "rule_2_applied": False, "rule_1_applied": False},
)

check("A01.40", "family A has all 95,441 cells", stats_all["n_cells"] == EXPECTED["n_cells"],
      detail=stats_all["n_cells"])
check("A01.41", "family A retains all 19 cell types",
      stats_all["n_celltypes"] == EXPECTED["n_celltypes"], detail=stats_all["celltypes"])
check("A01.42", "family A contains all 24 patients and both outcomes",
      stats_all["samples"] == PATIENTS and stats_all["outcomes"] == ["Good", "Poor"],
      detail={"n_samples": len(stats_all["samples"]), "outcomes": stats_all["outcomes"]})
check("A01.43", "family A per-cell-type counts equal the source annotation",
      stats_all["cells_per_celltype"] ==
      obs[CELLTYPE_COL].value_counts().sort_index().astype(int).to_dict(), detail=None)
{k: v for k, v in stats_all.items() if k != "cells_per_celltype"}

## 9. Family B - pooled balanced inputs (`inputs/pooled_seed{k}/{good,poor}/`)

In [ ]:
for seed in POOLED_SEEDS:
    sub = obs.loc[seed_selections[seed]]
    for group, patient_list in (("good", GOOD_PATIENTS), ("poor", POOR_PATIENTS)):
        outcome = group.capitalize()
        bcs = sub.index[sub[OUTCOME_COL] == outcome]
        stats = build_input(
            barcodes=bcs, out_dir=INPUTS_DIR / f"pooled_seed{seed}" / group,
            label=f"pooled_seed{seed}_{group}",
            expected_celltypes=retained_types,
            expected_samples=as_str_list(obs.loc[bcs, SAMPLE_COL].unique()),
            expected_outcomes=[outcome],
            provenance={"family": "B", "seed": seed, "group": group,
                        "is_primary": seed == PRIMARY_SEED,
                        "rule_2_applied": True, "rule_1_applied": True,
                        "rule_2_dropped": dropped_types,
                        "balance_target": {k: int(v) for k, v in BALANCE_TARGET.items()}},
        )
        check(f"A01.44.s{seed}.{group}", f"seed {seed} {group}: 42,214 cells / 15 cell types",
              stats["n_cells"] == EXPECTED["pooled_per_group_cells"]
              and stats["n_celltypes"] == EXPECTED["pooled_n_retained"],
              detail={"n_cells": stats["n_cells"], "n_celltypes": stats["n_celltypes"]})
        check(f"A01.45.s{seed}.{group}", f"seed {seed} {group}: only {outcome} cells present",
              stats["outcomes"] == [outcome], detail=stats["outcomes"])
        check(f"A01.46.s{seed}.{group}", f"seed {seed} {group}: only {outcome} patients present",
              set(stats["samples"]) <= set(patient_list),
              detail=as_str_list(set(stats["samples"]) - set(patient_list)))
        check(f"A01.47.s{seed}.{group}", f"seed {seed} {group}: no dropped cell type leaked in",
              not (set(stats["celltypes"]) & set(dropped_types)),
              detail=as_str_list(set(stats["celltypes"]) & set(dropped_types)))
        check(f"A01.48.s{seed}.{group}", f"seed {seed} {group}: per-type sizes hit the balance target",
              stats["cells_per_celltype"] == {k: int(v) for k, v in BALANCE_TARGET.sort_index().items()},
              detail=stats["cells_per_celltype"])

log_event({"event": "family_b_complete", "seeds": POOLED_SEEDS, "primary_seed": PRIMARY_SEED,
           "per_group_cells": POOLED_PER_GROUP, "retained": retained_types, "dropped": dropped_types})
sorted(k for k in BUILT if k.startswith("inputs/pooled"))

## 10. Family C - one input per patient (`inputs/per_patient/<SAMPLE>/`)

In [ ]:
for patient in PATIENTS:
    keep = patient_retained[patient]
    bcs = obs.index[(obs[SAMPLE_COL] == patient) & (obs[CELLTYPE_COL].isin(keep))]
    stats = build_input(
        barcodes=bcs, out_dir=INPUTS_DIR / "per_patient" / patient, label=f"per_patient_{patient}",
        expected_celltypes=keep, expected_samples=[patient],
        expected_outcomes=[patient_outcomes[patient]],
        min_cells_per_type=PATIENT_MIN_CELLS,
        provenance={"family": "C", "patient": patient, "outcome": patient_outcomes[patient],
                    "min_cells_per_celltype": PATIENT_MIN_CELLS,
                    "dropped_celltypes": as_str_list(set(ALL_CELLTYPES) - set(keep))},
    )
    per_type = pd.Series(stats["cells_per_celltype"])
    check(f"A01.49.{patient}", f"{patient}: only this patient's cells are present",
          stats["samples"] == [patient], detail=stats["samples"])
    check(f"A01.50.{patient}", f"{patient}: the single outcome label matches the derived mapping",
          stats["outcomes"] == [patient_outcomes[patient]], detail=stats["outcomes"])
    check(f"A01.51.{patient}", f"{patient}: every retained type has >= {PATIENT_MIN_CELLS} cells",
          int(per_type.min()) >= PATIENT_MIN_CELLS,
          detail=per_type[per_type < PATIENT_MIN_CELLS].to_dict())
    check(f"A01.52.{patient}", f"{patient}: retained types and counts match the abundance matrix",
          stats["celltypes"] == keep
          and per_type.sort_index().astype(int).to_dict()
              == abundance_ct.loc[keep, patient].sort_index().astype(int).to_dict(),
          detail={"written": per_type.to_dict(), "expected": abundance_ct.loc[keep, patient].to_dict()})

per_patient_dirs = sorted(p for p in (INPUTS_DIR / "per_patient").iterdir() if p.is_dir())
check("A01.53", "family C produced exactly 24 patient directories",
      len(per_patient_dirs) == EXPECTED["n_patients"], detail=len(per_patient_dirs))
check("A01.54", "family C directory names are exactly the derived patient IDs",
      sorted(p.name for p in per_patient_dirs) == PATIENTS,
      detail=sorted(p.name for p in per_patient_dirs))
log_event({"event": "family_c_complete", "n_dirs": len(per_patient_dirs),
           "retained_per_patient": {p: len(v) for p, v in patient_retained.items()}})

## 11. Whole-tree verification and checksums

Each directory was certified inside `build_input`; this pass re-enumerates the tree so that nothing
extra and nothing missing escapes notice.

In [ ]:
# ---- canonical per-sample x cell-type abundance table ---------------------------------
# Long form (batch, annot_6, n_cells). This is the shape the downstream notebooks resolve by
# column name (06_ groups by patient and sums n_cells; 07_ pivots it and asserts 24 x 19), so the
# canonical file is long and the wide view is derived from it rather than stored as the primary.
long_counts = (abundance_ct.rename_axis(CELLTYPE_COL).reset_index()
               .melt(id_vars=CELLTYPE_COL, var_name=SAMPLE_COL, value_name="n_cells")
               .loc[:, [SAMPLE_COL, CELLTYPE_COL, "n_cells"]]
               .sort_values([SAMPLE_COL, CELLTYPE_COL], kind="stable")
               .reset_index(drop=True))
long_counts["n_cells"] = long_counts["n_cells"].astype(int)
counts_csv = assert_safe_path(INPUTS_DIR / "per_sample_celltype_counts.csv")
long_counts.to_csv(counts_csv, index=False)
wide_counts = long_counts.pivot_table(index=SAMPLE_COL, columns=CELLTYPE_COL, values="n_cells",
                                      aggfunc="sum", fill_value=0).astype(int)

wide_csv = assert_safe_path(INPUTS_DIR / "per_sample_celltype_counts_wide_24x19.csv")
wide_counts.to_csv(wide_csv)

check("A01.64", "per_sample_celltype_counts.csv pivots to 24 patients x 19 cell types",
      wide_counts.shape == (EXPECTED["n_patients"], EXPECTED["n_celltypes"]),
      detail={"shape": wide_counts.shape, "n_rows_long": int(len(long_counts))})
check("A01.65", "per_sample_celltype_counts.csv totals 95,441 cells and matches the source obs",
      int(long_counts["n_cells"].sum()) == EXPECTED["n_cells"]
      and wide_counts.reindex(index=PATIENTS, columns=ALL_CELLTYPES).equals(
          pd.crosstab(obs[SAMPLE_COL], obs[CELLTYPE_COL])
            .reindex(index=PATIENTS, columns=ALL_CELLTYPES, fill_value=0).astype(int)),
      detail={"total": int(long_counts["n_cells"].sum())})
audit_copy = AUDIT_DIR / "per_sample_celltype_counts.csv"
check("A01.66", "the inputs/ abundance table is byte-identical to the copy 00_ wrote under audit/",
      (not audit_copy.is_file()) or sha256_file(audit_copy) == sha256_file(counts_csv),
      detail={"audit_copy": str(audit_copy), "exists": audit_copy.is_file()})
register_checksum(counts_csv)
register_checksum(wide_csv)

# ---- input tree ----------------------------------------------------------------------
expected_rel = sorted(
    [str((INPUTS_DIR / "all").relative_to(OUTPUT_ROOT))]
    + [str((INPUTS_DIR / f"pooled_seed{s}" / g).relative_to(OUTPUT_ROOT))
       for s in POOLED_SEEDS for g in ("good", "poor")]
    + [str((INPUTS_DIR / "per_patient" / p).relative_to(OUTPUT_ROOT)) for p in PATIENTS])
found_rel = sorted(str(p.parent.relative_to(OUTPUT_ROOT)) for p in INPUTS_DIR.rglob("counts.h5ad"))

check("A01.55", "the expected number of input directories exists (1 + 10 + 24 = 35)",
      len(expected_rel) == 1 + 2 * len(POOLED_SEEDS) + EXPECTED["n_patients"], detail=len(expected_rel))
check("A01.56", "no unexpected or missing input directory", found_rel == expected_rel,
      detail={"missing": sorted(set(expected_rel) - set(found_rel)),
              "unexpected": sorted(set(found_rel) - set(expected_rel))})
check("A01.57", "every input directory carries counts.h5ad, meta.tsv and retained_celltypes.json",
      all((OUTPUT_ROOT / d / f).is_file() for d in expected_rel
          for f in ("counts.h5ad", "meta.tsv", "retained_celltypes.json")),
      detail=[d for d in expected_rel
              if not all((OUTPUT_ROOT / d / f).is_file()
                         for f in ("counts.h5ad", "meta.tsv", "retained_celltypes.json"))])
check("A01.58", "every input directory was verified after being written",
      sorted(BUILT) == expected_rel, detail=sorted(set(expected_rel) ^ set(BUILT)))
check("A01.59", "a checksum was recorded for every written file "
                "(3 per input dir + the long and wide abundance tables)",
      len(CHECKSUMS) == 3 * len(expected_rel) + 2, detail=len(CHECKSUMS))
check("A01.60", "every written input uses the same gene space as the source",
      {v["n_genes"] for v in BUILT.values()} == {int(n_genes)},
      detail=sorted({v["n_genes"] for v in BUILT.values()}))
check("A01.67", "every written input declares the log1p CP10K expression contract",
      {v["expression_contract"] for v in BUILT.values()} == {COUNTS_CONTRACT},
      detail=sorted({v["expression_contract"] for v in BUILT.values()}))

build_table = pd.DataFrame([{"dir": k, "n_cells": v["n_cells"], "n_celltypes": v["n_celltypes"],
                             "n_samples": len(v["samples"]), "outcomes": ",".join(v["outcomes"]),
                             "counts_h5ad_sha256": v["counts_h5ad_sha256"]}
                            for k, v in sorted(BUILT.items())])
build_table.to_csv(AUDIT_DIR / "input_build_summary.csv", index=False)
pd.DataFrame({"path": list(CHECKSUMS), "sha256": list(CHECKSUMS.values())}).to_csv(
    AUDIT_DIR / "input_checksums.csv", index=False)

write_manifest([OUTPUT_ROOT / p for p in CHECKSUMS] + [AUDIT_DIR, LOG_DIR], MANIFEST_PATH)
check("A01.61", "MANIFEST.sha256 refreshed after the input build", MANIFEST_PATH.is_file(),
      detail=str(MANIFEST_PATH))
build_table


## Gate 1 - summary and STOP

`gate1_summary.md` records the derived patient lists, group sizes, the Rule 2 outcome, the pooled
balance per cell type, per-patient retention, the core cell-type set with its pair count, every
checksum and every assertion. **The pipeline stops here for human review.**

In [ ]:
failed_blocking = [c for c in CHECKS if not c["passed"] and c["blocking"]]
failed_soft = [c for c in CHECKS if not c["passed"] and not c["blocking"]]
n_core = len(CORE_CELLTYPES)
core_pairs_directed = n_core * n_core
core_pairs_unordered = n_core * (n_core + 1) // 2

L = ["# Gate 1 - CellPhoneDB input build (clinical_outcome_v2)\n",
     f"- Built: `{datetime.now(timezone.utc).isoformat()}`",
     f"- Config hash: `{CFG_HASH}`  |  source H5AD sha256: `{h5ad_sha}`",
     f"- Downsampling seeds: `{POOLED_SEEDS}` (primary = `{PRIMARY_SEED}`)  |  CellPhoneDB debug seed: `{CPDB_SEED}`",
     f"- Expression contract for every input: `{COUNTS_CONTRACT}` - raw `layers['{COUNTS_LAYER}']`"
     f" -> `sc.pp.normalize_total(target_sum={TARGET_SUM:g})` -> `sc.pp.log1p()`, applied once to the"
     f" full matrix in this notebook (`utils.write_cpdb_input` normalises nothing). Verified on disk"
     f" for every input; max library-size deviation tolerance {SCALE_ATOL:g}.",
     f"- Gene space: {n_genes:,} HGNC symbols (highly-variable subset; absence of an interaction is not evidence of absence)\n",
     "## Derived patient lists (never hardcoded)\n",
     f"**Good ({len(GOOD_PATIENTS)}):** " + ", ".join(f"`{p}`" for p in GOOD_PATIENTS),
     f"\n**Poor ({len(POOR_PATIENTS)}):** " + ", ".join(f"`{p}`" for p in POOR_PATIENTS),
     "\n| group | patients | cells |\n|---|---|---|",
     f"| Good | {len(GOOD_PATIENTS)} | {n_good_cells:,} |",
     f"| Poor | {len(POOR_PATIENTS)} | {n_poor_cells:,} |",
     f"| total | {len(PATIENTS)} | {n_good_cells + n_poor_cells:,} |\n",
     "## Rule 2 - cell types dropped from the pooled comparison\n",
     "| cell type | Good | Poor | min | status |\n|---|---|---|---|---|"]
for ct in ct_by_outcome.sort_values("min_group").index:
    row = ct_by_outcome.loc[ct]
    L.append(f"| {ct} | {int(row['Good']):,} | {int(row['Poor']):,} | {int(row['min_group']):,} | "
             f"{'**dropped**' if ct in dropped_types else 'retained'} |")
L.append(f"\nRetained **{len(retained_types)}/{len(ALL_CELLTYPES)}**; dropped exactly: "
         + ", ".join(f"`{d}`" for d in dropped_types) + ".\n")
L += ["### Rule 2 threshold reading (recorded decision)\n",
      f"The threshold `pooled_celltype_min_per_group = {POOLED_MIN_PER_GRP}` is applied "
      f"**exclusively** (`{RULE2['predicate_used']}`), via `{RULE2['utils_call']}` because "
      "`utils.pooled_retained_celltypes` implements the inclusive predicate.\n",
      "| reading | predicate | cell types retained | cells per group |\n|---|---|---|---|",
      f"| inclusive | `{RULE2['inclusive']['predicate']}` | {RULE2['inclusive']['n_retained']} | "
      f"{RULE2['inclusive']['cells_per_group']:,} |",
      f"| **exclusive (used)** | `{RULE2['exclusive']['predicate']}` | "
      f"**{RULE2['exclusive']['n_retained']}** | **{RULE2['exclusive']['cells_per_group']:,}** |",
      f"\nBoundary cell types (exactly {POOLED_MIN_PER_GRP} in the smaller group): "
      + (", ".join(f"`{b}`" for b in RULE2["boundary_celltypes"]) or "none") + ".\n",
      "**Deliberate resolution of a boundary case - do not revert this to `>=`.** The plan "
      "documents contradict themselves: they state the rule as `min < 10 -> drop`, which would "
      "*retain* `Plasma cells` (10 Good / 157 Poor), while simultaneously listing `Plasma cells` "
      "among the drops and quoting 15 types / 42,214 cells per group, which requires `> 10`. The "
      "gap is exactly 10 cells - Plasma cells' own minimum. The decision taken is to drop it: "
      "balancing would downsample Poor from 157 to 10, resting the whole comparison for that type "
      "on 10 cells per group, and Rule 2 exists precisely to remove presence/absence artefacts.\n",
      f"**Fragility note.** `{EXPECTED['pooled_smallest_retained']}` "
      f"({int(BALANCE_TARGET[EXPECTED['pooled_smallest_retained']])} cells per group after "
      "balancing) is retained under either threshold, but it is the smallest retained type by an "
      "order of magnitude. No claim should rest on it.\n"]

L += [f"## Rule 1 - pooled balance (seed {PRIMARY_SEED})\n",
      "| cell type | target per group | Good written | Poor written |\n|---|---|---|---|"]
tab0 = pooled_tables[PRIMARY_SEED]
for ct in retained_types:
    L.append(f"| {ct} | {int(BALANCE_TARGET[ct]):,} | {int(tab0.loc[ct, 'Good']):,} | "
             f"{int(tab0.loc[ct, 'Poor']):,} |")
L.append(f"| **total** | **{POOLED_PER_GROUP:,}** | **{int(tab0['Good'].sum()):,}** | "
         f"**{int(tab0['Poor'].sum()):,}** |")
L.append(f"\nBalanced pooled set = {2 * POOLED_PER_GROUP:,} cells "
         f"({2 * POOLED_PER_GROUP / EXPECTED['n_cells']:.1%} of {EXPECTED['n_cells']:,}). "
         f"Seeds {POOLED_SEEDS[1:]} reproduce the same sizes with different cells.\n")

L += [f"## Family C - per-patient retention (>= {PATIENT_MIN_CELLS} cells per type)\n",
      "| patient | outcome | cells written | retained types |\n|---|---|---|---|"]
for p in sorted(PATIENTS, key=lambda x: n_ret[x]):
    st = BUILT[str((INPUTS_DIR / "per_patient" / p).relative_to(OUTPUT_ROOT))]
    L.append(f"| {p} | {patient_outcomes[p]} | {st['n_cells']:,} | {st['n_celltypes']} |")
L.append(f"\nRetained types per patient: min {int(n_ret.min())}, median {n_ret.median():g}, "
         f"max {int(n_ret.max())}.\n")

L += ["## Core cell-type set (secondary analysis)\n",
      f"`min_cells = {PATIENT_MIN_CELLS}`, `min_patients = {CORE_MIN_PATIENTS}` -> **{n_core} cell types**: "
      + ", ".join(f"`{c}`" for c in CORE_CELLTYPES),
      f"\nInteraction pairs on the core set: **{core_pairs_directed} directed** "
      f"({core_pairs_unordered} unordered including self-pairs).\n",
      "## Inputs written\n",
      "| directory | cells | cell types | samples | outcomes |\n|---|---|---|---|---|"]
for k, v in sorted(BUILT.items()):
    L.append(f"| `{k}` | {v['n_cells']:,} | {v['n_celltypes']} | {len(v['samples'])} | "
             f"{','.join(v['outcomes'])} |")

L += ["## Per-sample cell-type abundance\n",
      f"`{counts_csv.relative_to(OUTPUT_ROOT)}` - long form, {len(long_counts):,} rows "
      f"({wide_counts.shape[0]} patients x {wide_counts.shape[1]} cell types), "
      f"{int(long_counts['n_cells'].sum()):,} cells total.\n",
      "\n## Checksums (sha256)\n", "| file | sha256 |\n|---|---|"]
L += [f"| `{k}` | `{v}` |" for k, v in sorted(CHECKSUMS.items())]

L += ["\n## Assertions\n",
      f"{len(CHECKS)} checks; {len(failed_blocking)} blocking failures; "
      f"{len(failed_soft)} non-blocking flags.\n",
      "| id | result | description |\n|---|---|---|"]
for c in CHECKS:
    L.append(f"| `{c['id']}` | {'PASS' if c['passed'] else ('FAIL' if c['blocking'] else 'FLAG')} | "
             f"{c['description']} |")
L.append("\n---\n**STOP.** Gate 1 requires human review before any CellPhoneDB run is launched.\n")

assert_safe_path(GATE1_MD).write_text("\n".join(L) + "\n")
GATE1_MD

In [ ]:
# ---- Gate 1 explicit final assertions ------------------------------------------------
assert len(GOOD_PATIENTS) == 13 and len(POOR_PATIENTS) == 11, "patient split"
assert (n_good_cells, n_poor_cells) == (46260, 49181), "cell split"
assert stats_all["n_cells"] == 95441 and stats_all["n_celltypes"] == 19, "family A"
assert dropped_types == sorted(EXPECTED["pooled_dropped_celltypes"]), "Rule 2 dropped set"
assert len(retained_types) == 15, "Rule 2 retained count"
assert POOLED_PER_GROUP == 42214, "Rule 1 balanced group size"
assert wide_counts.shape == (24, 19), "per_sample_celltype_counts.csv shape"
assert observed_scale == COUNTS_CONTRACT == "log1p_cp10k", "expression contract"
assert all(BUILT[f"inputs/pooled_seed{s}/{g}"]["n_cells"] == 42214
           for s in POOLED_SEEDS for g in ("good", "poor")), "pooled input sizes"
assert len(BUILT) == 1 + 2 * len(POOLED_SEEDS) + 24, "input directory count"
assert not failed_blocking, f"blocking checks failed: {[c['id'] for c in failed_blocking]}"

GATE1 = {
    "gate": "gate1_inputs",
    "notebook": "01_build_cpdb_inputs",
    "passed": True,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "config_hash": CFG_HASH,
    "input_h5ad_sha256": h5ad_sha,
    "downsample_seeds": POOLED_SEEDS,
    "primary_seed": PRIMARY_SEED,
    "cpdb_debug_seed": CPDB_SEED,
    "expression_contract": COUNTS_CONTRACT,
    "target_sum": TARGET_SUM,
    "good_patients": GOOD_PATIENTS, "poor_patients": POOR_PATIENTS,
    "n_good_cells": n_good_cells, "n_poor_cells": n_poor_cells,
    "rule2_retained": retained_types, "rule2_dropped": dropped_types,
    "rule2_threshold_reading": RULE2,
    "per_sample_celltype_counts": str(counts_csv.relative_to(OUTPUT_ROOT)),
    "balance_target": {k: int(v) for k, v in BALANCE_TARGET.items()},
    "pooled_cells_per_group": POOLED_PER_GROUP,
    "per_patient_retained": patient_retained,
    "core_celltypes": CORE_CELLTYPES,
    "core_pairs_directed": core_pairs_directed,
    "inputs": {k: {kk: vv for kk, vv in v.items() if kk != "cells_per_celltype"}
               for k, v in sorted(BUILT.items())},
    "checksums": CHECKSUMS,
    "n_checks": len(CHECKS),
    "n_failed_blocking": len(failed_blocking),
    "n_failed_soft": len(failed_soft),
    "soft_failures": [c["id"] for c in failed_soft],
    "checks": CHECKS,
    "gate1_summary": str(GATE1_MD.relative_to(OUTPUT_ROOT)),
}
gate1_path = assert_safe_path(AUDIT_DIR / "gate1_success.json")
gate1_path.write_text(json.dumps(GATE1, indent=2, default=str))
log_event({"event": "gate1", "passed": True, "n_checks": len(CHECKS),
           "soft_failures": [c["id"] for c in failed_soft]})

print("=" * 78)
print(f"GATE 1 PASSED - {len(BUILT)} input directories, {len(CHECKS)} checks, "
      f"{len(failed_soft)} non-blocking flags {[c['id'] for c in failed_soft]}")
print(f"Review {GATE1_MD} before running CellPhoneDB. STOP HERE.")
print("=" * 78)
list(GATE1["inputs"])